# Day 6: Data Quality, Validation & Quarantine DLQ Engine

This notebook demonstrates validation rules and DLQ (Dead Letter Queue) routing for bad records.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp

spark = SparkSession.builder \
    .appName("Day06_DQ_DLQ") \
    .getOrCreate()

print("Spark session initialized.")

In [ ]:
data = [
    ("P001", 75, "2023-05-01"),  # Good
    (None, 80, "2023-05-02"),    # Bad: Null patient_id
    ("P002", 999, "2023-05-03"), # Bad: Implausible heart rate
]
columns = ["patient_id", "vitals_heart_rate", "measurement_date"]
df_raw = spark.createDataFrame(data, columns)

print("Raw Data:")
df_raw.show()

In [ ]:
good_df = df_raw.filter(
    col("patient_id").isNotNull() & 
    (col("vitals_heart_rate").between(30, 250) | col("vitals_heart_rate").isNull())
)

bad_df = df_raw.filter(
    col("patient_id").isNull() | 
    ~col("vitals_heart_rate").between(30, 250)
).withColumn("dq_error_manifest", lit("Failed structural or clinical rules")) \
 .withColumn("quarantine_timestamp", current_timestamp())

print("Good Records (Proceed to Silver):")
good_df.show()

print("DLQ Records (Routed to Quarantine):")
bad_df.show(truncate=False)